In [6]:
import pandas as pd
from pathlib import Path

In [7]:
DATA_PATH = Path("../data/raw")

train_df = pd.read_csv(DATA_PATH / "cs-training.csv")
train_df = train_df.drop(columns="Unnamed: 0")

In [8]:
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin


class CreditPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.monthly_income_median_ = X["MonthlyIncome"].median()
        return self

    def transform(self, X):
        X = X.copy()

        # Missing MonthlyIncome indicator
        X["MissingMI"] = X["MonthlyIncome"].isna().astype(int)

        # Impute MonthlyIncome with median
        X["MonthlyIncome"] = X["MonthlyIncome"].fillna(
            self.monthly_income_median_
        )

        # Impute NumberOfDependents with 0
        X["NumberOfDependents"] = X["NumberOfDependents"].fillna(0)

        return X

In [9]:
preprocessor = CreditPreprocessor()

X_train = train_df.drop(columns="SeriousDlqin2yrs")
y_train = train_df['SeriousDlqin2yrs']
X_train_processed = preprocessor.fit_transform(X_train)

In [10]:
# When we take a look at MonthlyIncome and NumberOfDependants we see information missing
X_train_processed.info()


<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 1   age                                   150000 non-null  int64  
 2   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 3   DebtRatio                             150000 non-null  float64
 4   MonthlyIncome                         150000 non-null  float64
 5   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 6   NumberOfTimes90DaysLate               150000 non-null  int64  
 7   NumberRealEstateLoansOrLines          150000 non-null  int64  
 8   NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 9   NumberOfDependents                    150000 non-null  float64
 10  MissingMI                             150000 non-null  int64  
dtypes: float64(

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

preprocessor = Pipeline([
    ("credit", CreditPreprocessor()),
    ("scaler", StandardScaler()),
])

X_train_processed = preprocessor.fit_transform(X_train)

In [12]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_processed,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42,
)

In [19]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [None, 3, 5, 7, 10],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [10, 20, 50, 100],
    "l2_regularization": [0, 0.01, 0.1, 1],
    "max_iter": [100, 200, 300, 500],
}

search = RandomizedSearchCV(
    estimator=HistGradientBoostingClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=40,
    scoring="roc_auc",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=2,
)

search.fit(X_tr, y_tr)

print("Best ROC AUC:", search.best_score_)
print("Best params:")
print(search.best_params_)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best ROC AUC: 0.864121516246777
Best params:
{'min_samples_leaf': 100, 'max_leaf_nodes': 15, 'max_iter': 100, 'max_depth': 10, 'learning_rate': 0.1, 'l2_regularization': 0}


In [21]:
best_model = search.best_estimator_

from sklearn.calibration import CalibratedClassifierCV

calibrated_model = CalibratedClassifierCV(
    best_model,
    method="isotonic",
    cv=3,
)

calibrated_model.fit(X_tr, y_tr)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",HistGradientB...ndom_state=42)
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'isotonic'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary <n_jobs>` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
Name,Type,Value
"calibrated_classifiers_ calibrated_classifiers_: list (len() equal to cv or 1 if `ensemble=False`)The list of classifier and calibrator pairs.- When `ensemble=True`, `n_cv` fitted `estimator` and calibrator pairs. `n_cv` is the number of cross-validation folds.- When `ensemble=False`, the `estimator`, fitted on all the data, and fitted calibrator... versionchanged:: 0.24 Single calibrated classifier case when `ensemble=False`.",list,"[<sklearn.cali...001BA4CA807D0>, <s

In [23]:
y_proba = calibrated_model.predict_proba(X_val)[:, 1]
y_pred = calibrated_model.predict(X_val)
threshold = 0.737
y_pred = (y_proba >= threshold).astype(int)

In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

print(f"Accuracy : {accuracy_score(y_val, y_pred):.4f}")
print(f"Precision: {precision_score(y_val, y_pred):.4f}")
print(f"Recall   : {recall_score(y_val, y_pred):.4f}")
print(f"F1-score : {f1_score(y_val, y_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_val, y_proba):.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_val, y_pred))

print("\nClassification Report")
print(classification_report(y_val, y_pred))

Accuracy : 0.9339
Precision: 0.7619
Recall   : 0.0160
F1-score : 0.0313
ROC AUC  : 0.8684

Confusion Matrix
[[27985    10]
 [ 1973    32]]

Classification Report
              precision    recall  f1-score   support

           0       0.93      1.00      0.97     27995
           1       0.76      0.02      0.03      2005

    accuracy                           0.93     30000
   macro avg       0.85      0.51      0.50     30000
weighted avg       0.92      0.93      0.90     30000



In [26]:
import numpy as np
from sklearn.metrics import f1_score

y_proba = calibrated_model.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.01, 0.99, 99)

best_threshold = 0.5
best_f1 = 0

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    f1 = f1_score(y_val, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"Best threshold: {best_threshold:.2f}")
print(f"Best F1: {best_f1:.4f}")

Best threshold: 0.21
Best F1: 0.4470
